# Laser Off Code

In [ ]:
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

# Import

In [1]:
import time
from time import sleep, monotonic
import datetime
import numpy as np
import matplotlib.pyplot as plt
import sys
import pyvisa
import qcodes as qc
from qcodes.dataset import Measurement
from qcodes.dataset import do0d
from qcodes.dataset.experiment_container import new_experiment, load_experiment_by_name
from qcodes.dataset.plotting import plot_by_id
from qcodes.dataset.data_set import load_by_id, load_by_counter
from qcodes import initialise_or_create_database_at, new_data_set, new_experiment
from qcodes.station import Station
initialise_or_create_database_at("./2026-03-10_SNSPD4.db")
from functions import osc_set_standard, osc_check_standard, capture_trace, capture_trace_simple
import snspd 
params = snspd.snspd()

# Set up experiment
exp_name = 'SNSPD4_23_03_2026'
sample_name = '00'

try:
    exp = qc.load_experiment_by_name(exp_name, sample=sample_name)
    print('Experiment loaded. Last ID no:', exp.last_counter)
except ValueError:
    exp = new_experiment(exp_name, sample_name)
    print('Started new experiment')

Logging hadn't been started.
Activating auto-logging. Current session state plus future input saved.
Filename       : C:\Users\QNL\.qcodes\logs\command_history.log
Mode           : append
Output logging : True
Raw input log  : False
Timestamping   : True
State          : active
Qcodes Logfile : C:\Users\QNL\.qcodes\logs\260415-31316-qcodes.log
Experiment loaded. Last ID no: 483


# Instruments

In [3]:
station = Station(config_file="friesland.yaml")
dmm = station.load_instrument("dmm", revive_instance=True)
yoko = station.load_instrument("yoko", revive_instance=True)
laser = station.load_instrument("laser", revive_instance=True)
MS = station.load_instrument("mso5", revive_instance=True)
pm100usb = station.load_instrument("pm100usb", revive_instance=True)
pms120 = station.load_instrument("pms120", revive_instance=True)
tc = station.load_instrument("fridge", revive_instance=True)
p_att = station.load_instrument("dmm_keithley", revive_instance=True) # excluding from snapshot because none of the parameters work anyway

2026-04-15 12:55:34,233 ¦ qcodes.instrument.instrument_base.com.visa ¦ ERROR ¦ visa ¦ _connect_and_handle_error ¦ 222 ¦ [laser(PPCL550)] Could not connect at ASRL13
Traceback (most recent call last):
  File "C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\qcodes\instrument\visa.py", line 218, in _connect_and_handle_error
    visa_handle, visabackend, resource_manager = self._open_resource(
                                                 ~~~~~~~~~~~~~~~~~~~^
        address, visalib
        ^^^^^^^^^^^^^^^^
    )
    ^
  File "C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\qcodes\instrument\visa.py", line 246, in _open_resource
    resource = resource_manager.open_resource(address)
  File "C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\highlevel.py", line 3302, in open_resource
    res.open(access_mode, open_timeout)
    ~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\resource.py", line 290, in op

VisaIOError: VI_ERROR_RSRC_BUSY (-1073807246): The resource is valid, but VISA cannot currently access it.

# Trace Captures

Old Device 

Set trigger at a lower level, base on set thresholds 

Parameters from SNSPD4_5_Counts_vs_Current.ipynb
v_attenuator = 4.7
v_scale_old_device = 50e-3
current_range = np.arange(1e-7, 4.1e-6, 1e-7)
device_name = 'Line 2 Old Device'

In [ ]:
# Parameters from SNSPD4_1_Trace_Capture.ipynb
h_time = 3e-6 # time for trace acquisition
h_pos = 10
v_scale=150e-3
v_trigger = -200e-3

In [ ]:
from functions import set_thresholds
v_scale_old_device = 50e-3
osc_set_standard(MS, v_scale=v_scale_old_device, h_time=params.h_time_counts, h_pos=params.h_pos_counts)
threshold1, threshold2 = set_thresholds(yoko.current()) 

# try trigger at each level 
v_trigger 

osc_set_standard(MS, v_scale=v_scale_old_device, h_time=params.h_time_counts, h_pos=params.h_pos_counts)

In [17]:
capture_trace(MS, dmm, p_att)

Starting experimental run with id: 321. 
321
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 38.4V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;1.120E-9;188;"V";6.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1


In [ ]:
capture_trace(MS, dmm, p_att, station=True)

In [18]:
osc_set_standard(MS, v_trigger = v_trigger, v_scale=v_scale, h_time=h_time, h_pos=h_pos)

Running again with proper shielding (box closed, fibre wrapped)

In [35]:
print(v_scale) 
print(h_time)
print(h_pos)
print(v_trigger)

osc_set_standard(MS, v_trigger = v_trigger, v_scale=v_scale, h_time=h_time, h_pos=h_pos)

if MS.channels[0].clipping(): 
    print('Error: Clipping')


############################ TURN LASER ON ############################ 
laser.enable(True)

time.sleep(10)
print(f'Laser enable status: {laser.enable()}')


for v in np.arange(3.4, 6.1, 0.1)[::-1]:

    # Set attenuator voltage 
    p_att.write(f'VOLT {v}')

    time.sleep(20) # to let the trace change because of that weird thing that happens with the trace not adjusting immediately

    print(f'Voltage on attenuator is {p_att.ask('VOLT?')}')

    for i in range(2):

        # capture the trace 
        capture_trace(MS, dmm, p_att, station=station)

        time.sleep(5)
    
    
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

0.15
3e-06
10
-0.2
Laser enable status: True
Voltage on attenuator is 6


2026-04-10 11:52:20,851 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()

2026-04-10 11:52:36,740 ¦ asyncio ¦ ERROR ¦ base_events ¦ default_exception_handler ¦ 1879 ¦ Task was destroyed but it is pending!
task: <Task pending name='Task-238' coro=<_async_in_context.<locals>.run_in_context() done, defined at C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\ipykernel\utils.py:57> wait_for=<Task pending name='Task-239' coro=<Kernel.shell_main() running at C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\zmq\eventloop\zmqstream.py:563]>
2026-04-10 11:52:36,742 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\env

Starting experimental run with id: 385. 
385
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 38.4V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;800.0E-12;188;"V";6.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1


2026-04-10 11:52:43,063 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Starting experimental run with id: 386. 
386
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 38.4V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;640.00E-12;188;"V";6.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1
Voltage on attenuator is 5.9


...

Starting experimental run with id: 436. 
436
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 38.4V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;395.000E-12;188;"V";6.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1
Voltage on attenuator is 3.4


2026-04-10 12:21:22,212 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Starting experimental run with id: 437. 
437
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 38.4V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;320.00E-12;188;"V";6.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1


2026-04-10 12:21:44,691 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Starting experimental run with id: 438. 
438
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 38.4V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;640.00E-12;188;"V";6.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1
Laser enable status: False
